# Modelling — FX Activation

One notebook per product: a thin caller of `src/run.py` plus inspection. Reads the narrower `cfg.features` (candidates − exclusions). All logic lives in `src/`.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load

### 0 · (Optional) smoke-test the plumbing
Confirm FLAML works on throwaway synthetic data before the real run. No domain claim.

In [ ]:
import numpy as np, pandas as pd
from flaml import AutoML
rng = np.random.default_rng(0); n = 4000
Xs = pd.DataFrame({f'f{i}': rng.normal(size=n) for i in range(6)})
ys = (Xs['f0'] + 0.3 * rng.normal(size=n) > 0).astype(int)
m = AutoML(); m.fit(X_train=Xs, y_train=ys, task='classification', metric='roc_auc',
                    time_budget=10, estimator_list=['lgbm'], eval_method='cv',
                    n_splits=3, verbose=0)
print('best estimator:', m.best_estimator, '| plumbing OK')

### 1 · The real run
Loads months (pruned in Spark), builds labels, asserts the contract, writes the leakage report, optionally downsamples the train frame (calibration auto-applies), fits FLAML on curated features, evaluates OOT, computes permutation importance, scores the inference month, and writes artifacts.

In [ ]:
from src.run import run
result = run(cfg, spark)
result['run_id'], result['out']

### 2 · OOT metrics
AUC plus the operational metrics: precision / recall / lift at the **top 1%** and **top decile** of the ranking — what matters when you can only contact a slice of a large base. Lift reference is 1.0 (= base rate).

In [ ]:
import json
result['metrics']

### 3 · Winning model
The best estimator type and its tuned hyperparameters (`best_config`), plus the best CV loss — recorded in the card so the artifact is self-describing without unpickling.

In [ ]:
import pathlib
out = pathlib.Path(result['out'])
json.loads((out / 'model_card.json').read_text())['model_selected']

### 4 · Model-agnostic feature importance
Permutation importance on OOT (AUC drop when a feature is shuffled). Works for any model family FLAML picks.

In [ ]:
import pandas as pd
pd.read_csv(out / 'permutation_importance.csv').head(15)

### 5 · Inspect the rest of the run

In [ ]:
pd.read_csv(out / 'leakage_report.csv').head(10)

In [ ]:
json.loads((out / 'model_card.json').read_text())['caveats']

In [ ]:
s = pd.read_parquet(out / 'scores.parquet'); print(s.shape); s.head()

### 6 · Before trusting anything, eyeball
1. Base rate plausible?  2. Top-1% / top-decile **precision & recall** usable for the campaign size?  3. Any `suspected_leak` or non-persistent feature ranking high in importance — back to Feature Checks.  4. If downsampled, `score` is calibrated, `score_raw` is not.